# A2A Researcher 서버 — 정답본

원본: `src/solution/Chapter02/02-a2a/01_a2a_server.py`  
슬라이드: `docs/raw/Chapter02/day01/3-3-a2a-protocol-understanding.pdf`

## 학습 목표

- Agent Card·AgentExecutor·EventQueue 역할
- 원본 코드를 CodeMate와 함께 읽고 실행 결과를 점검한다.


## 코드 흐름

Agent Card 공개 → JSON-RPC → RequestContext → LangGraph → EventQueue

**읽을 때 확인할 점:** search_web은 모의 검색이다. a2a-sdk 버전과 코드의 타입·메서드가 맞는지 확인한다.


## 실행 전 확인

Jupyter 실행 루프와 서버 실행이 충돌하지 않도록 setup 셀이 `__name__`을 설정한다. 원본 코드 셀은 앱을 정의하지만 서버를 시작하지 않는다. 저장소 루트의 별도 터미널에서 `uv run python src/solution/Chapter02/02-a2a/01_a2a_server.py`로 포트 8001 서버를 시작한 뒤 클라이언트를 실행한다.

아래 원본 코드 셀은 실행을 직접 선택해야 하는 `manual-run` 셀입니다. 실행 준비 셀은 프로젝트 루트를 찾고 원본 `.py`의 경로를 `__file__`에 설정합니다.


In [ ]:
from pathlib import Path
import os
import sys

_repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "Chapter02").is_dir()),
    None,
)
if _repo_root is None:
    raise FileNotFoundError("저장소 안에서 이 노트북을 열어 주세요.")
os.chdir(_repo_root)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root / "src"))
__file__ = str(_repo_root / "src/solution/Chapter02/02-a2a/01_a2a_server.py")
# Jupyter에서는 스크립트 전용 __main__ 분기를 실행하지 않습니다.
__name__ = "notebook"


## 원본 코드

다음 셀에는 원본 `.py` 전체가 그대로 들어 있습니다.


In [ ]:
"""
A2A 서버: LangGraph Agent를 A2A 프로토콜로 노출하기

A2A(Agent-to-Agent)는 Google이 제안한 에이전트 간 통신 표준이다.
에이전트를 HTTP 서비스로 노출하면 프레임워크에 관계없이 (LangGraph, CrewAI, AutoGen 등)
서로 통신할 수 있다.

실행:
  python 01_a2a_server.py   (포트 8001)

확인:
  curl http://localhost:8001/.well-known/agent-card.json
"""

import asyncio
import uuid
from typing import Annotated

import a2a.types as t
import uvicorn
# AgentExecutor: 실질적으로 에이전트를 실행하도록 하는 주체
    # 내부적으로 정의해둔 랭그래프 에이전트를 실행
# RequestContext
    # task, message | 메시지 자체를 발행하고 처리하는 로직
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers.default_request_handler_v2 import DefaultRequestHandlerV2
from a2a.server.routes.agent_card_routes import create_agent_card_routes
from a2a.server.routes.jsonrpc_routes import create_jsonrpc_routes
from a2a.server.tasks import InMemoryTaskStore
from a2a.server.events.in_memory_queue_manager import InMemoryQueueManager
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from starlette.applications import Starlette
from starlette.routing import Route
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()


# ── 내부 LangGraph Agent ──────────────────────────────────────────────────────

@tool
def search_web(query: str) -> str:
    """웹에서 정보를 검색한다."""
    return f'"{query}" 검색 결과: 해당 주제에 대한 최신 정보 (데모 데이터)'


class State(TypedDict):
    messages: Annotated[list, add_messages]


llm = get_llm().bind_tools([search_web])
tool_node = ToolNode([search_web])


def agent_node(state: State) -> dict:
    return {'messages': [llm.invoke(state['messages'])]}


builder = StateGraph(State)
builder.add_node('agent', agent_node)
builder.add_node('tools', tool_node)
builder.add_edge(START, 'agent')
builder.add_conditional_edges('agent', tools_condition)
builder.add_edge('tools', 'agent')
langgraph_agent = builder.compile()


# ── A2A AgentExecutor ─────────────────────────────────────────────────────────
# AgentExecutor.execute(): RequestContext에서 사용자 메시지를 읽어
# EventQueue에 결과(Message)를 발행한다.
#
# a2a-sdk 1.x에서는 Pydantic 대신 protobuf 타입(a2a.types)을 사용한다.
# protobuf는 스키마를 .proto 파일로 정의하고, 언어별 코드를 자동 생성하는 직렬화 포맷이다.
# gRPC 기반의 크로스 플랫폼 통신을 위해 SDK가 protobuf로 전환됐다.

class ResearcherExecutor(AgentExecutor):
    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        # context.get_user_input(): 사용자 메시지의 텍스트 파트를 모아 문자열로 반환
        user_text = context.get_user_input() or ''

        result = await langgraph_agent.ainvoke({
            'messages': [HumanMessage(content=user_text)]
        })
        answer = result['messages'][-1].content

        # t.Message: protobuf 기반 응답 메시지
        # t.Part(text=...): 텍스트 파트 (1.x에서는 TextPart 대신 Part.text 직접 사용)
        await event_queue.enqueue_event(
            t.Message(
                role=t.Role.ROLE_AGENT,
                parts=[t.Part(text=answer)],
                message_id=str(uuid.uuid4()),
                task_id=context.task_id,
                context_id=context.context_id,
            )
        )

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        pass


# ── Agent Card ────────────────────────────────────────────────────────────────
# Agent Card: 이 에이전트가 무엇을 할 수 있는지 설명하는 메타데이터.
# 클라이언트는 /.well-known/agent-card.json에서 이를 읽어 에이전트를 발견(discover)한다.

agent_card = t.AgentCard(
    name='Researcher Agent',
    description='주어진 주제를 웹에서 조사하고 요약 정보를 제공합니다.',
    version='1.0.0',
    default_input_modes=['text'],
    default_output_modes=['text'],
    capabilities=t.AgentCapabilities(streaming=False),
    supported_interfaces=[t.AgentInterface(url='http://localhost:8001')],
    skills=[
        t.AgentSkill(
            id='web_research',
            name='웹 리서치',
            description='주제를 검색해 핵심 정보를 수집하고 정리합니다.',
            tags=['research', 'web'],
            examples=['AI Agent 기술 트렌드 조사', 'LangGraph 최신 기능 알려줘'],
        )
    ],
)

# ── Starlette 앱 조립 ─────────────────────────────────────────────────────────

queue_manager = InMemoryQueueManager()
task_store = InMemoryTaskStore()

request_handler = DefaultRequestHandlerV2(
    agent_executor=ResearcherExecutor(),
    task_store=task_store,
    agent_card=agent_card,
    queue_manager=queue_manager,
)

routes: list[Route] = [
    *create_agent_card_routes(agent_card),
    *create_jsonrpc_routes(request_handler, rpc_url='/'),
]

app = Starlette(routes=routes)

if __name__ == '__main__':
    print('Researcher A2A 서버 시작: http://localhost:8001')
    print('Agent Card: http://localhost:8001/.well-known/agent-card.json')
    uvicorn.run(app, host='0.0.0.0', port=8001)


## CodeMate와 함께 검토

> Agent Card의 설명과 execute의 동작을 대조하고 요청부터 EventQueue 발행까지 흐름을 설명해 줘. 원본은 바꾸지 말고 검토 내용을 먼저 말해 줘.

제안된 변경은 어떤 문제를 해결하는지, 어떤 결과로 확인할지 설명해 보세요.

## 확인 질문

1. 클라이언트가 Agent Card를 먼저 읽는 이유는?
2. EventQueue의 결과는 어디서 만드는가?
